スタッキングモデルを作る。
xgboost+DNNで予測値の推定を行う。


In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn.model_selection import train_test_split, KFold

In [6]:
california = fetch_california_housing()
df = pd.DataFrame(california.data, columns=california.feature_names)
df['Target'] = california.target
X = df.drop('Target', axis=1).values
y = df['Target'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

今からすることは、ハイパパラメータのチューニングである。
これは、k-foldしている最中に、それぞれのモデルでやるのが理想だが、計算コストの観点から厳しいので、訓練データで最適なハイパパラメータのチューニングをする。

In [3]:
from sklearn.metrics import mean_squared_error
import optuna

In [7]:
def objective(trial):
    param = {
        'random_state': 42,
        'eval_metric': 'rmse',
        # 木の深さ (3〜9の整数)
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        # 学習率 (0.01〜0.3の範囲で対数的に探索)
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        # データの抽出割合 (0.6〜1.0)
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        # 特徴量の抽出割合 (0.6〜1.0)
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        # L2正則化 (1e-3〜10.0の範囲で対数的に探索)
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        
        # アーリーストッピングを効かせるため、木の数は大きめに設定
        'n_estimators': 2000 
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_rmse = []
    for train_idx, valid_idx in kf.split(X_train):
        X_tr, y_tr = X_train[train_idx], y_train[train_idx]
        X_va, y_va = X_train[valid_idx], y_train[valid_idx]
        
        # アーリーストッピングを設定
        model = xgb.XGBRegressor(**param, early_stopping_rounds=50)
        
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False            
        )
        
        preds = model.predict(X_va)
        rmse = np.sqrt(mean_squared_error(y_va, preds))
        cv_rmse.append(rmse)

    return np.mean(cv_rmse)

In [8]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)
print(f"▼ ベストスコア (RMSE): {study.best_value:.4f}")
print("▼ 最も精度の高かったパラメータ:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-05-31 03:59:47,275] A new study created in memory with name: no-name-29fffdb4-96b5-4814-9d41-b2798a989027
[I 2026-05-31 03:59:48,552] Trial 0 finished with value: 0.4743570093431435 and parameters: {'max_depth': 3, 'learning_rate': 0.19862986020278867, 'subsample': 0.8161675735418924, 'colsample_bytree': 0.743382564361267, 'reg_lambda': 2.4297887122059096}. Best is trial 0 with value: 0.4743570093431435.
[I 2026-05-31 03:59:49,887] Trial 1 finished with value: 0.46513376440298354 and parameters: {'max_depth': 4, 'learning_rate': 0.1423359560848573, 'subsample': 0.7306060320465434, 'colsample_bytree': 0.7983495873101314, 'reg_lambda': 0.004679931210615799}. Best is trial 1 with value: 0.46513376440298354.
[I 2026-05-31 03:59:50,602] Trial 2 finished with value: 0.48005305338228865 and parameters: {'max_depth': 4, 'learning_rate': 0.2986287100946006, 'subsample': 0.8864146290685031, 'colsample_bytree': 0.7030256297409365, 'reg_lambda': 0.019086395264000817}. Best is trial 1 with 

▼ ベストスコア (RMSE): 0.4444
▼ 最も精度の高かったパラメータ:
  max_depth: 7
  learning_rate: 0.01558243137576329
  subsample: 0.8085482113664683
  colsample_bytree: 0.7928410517065787
  reg_lambda: 0.026532004962529276


比較対象のために、xgboostのチューニングされたハイパパラメータのもとでのRMSEを出しておく。

In [9]:
best_params = study.best_params
best_params['random_state'] = 42
best_params['eval_metric'] = 'rmse'
best_params['n_estimators'] = 2000
X_tr, X_va, y_tr, y_va = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)
best_model = xgb.XGBRegressor(**best_params,early_stopping_rounds=50)
best_model.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)], # 切り出した監視用データを渡す
    verbose=100              # 100本ごとにログを表示 (邪魔なら False に)
)

test_preds = best_model.predict(X_test)
final_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
print(f"\n▼ テストデータでのRMSE: {final_rmse:.4f}")


[0]	validation_0-rmse:1.16263
[100]	validation_0-rmse:0.60728
[200]	validation_0-rmse:0.51886
[300]	validation_0-rmse:0.49373
[400]	validation_0-rmse:0.48320
[500]	validation_0-rmse:0.47801
[600]	validation_0-rmse:0.47505
[700]	validation_0-rmse:0.47179
[800]	validation_0-rmse:0.46941
[900]	validation_0-rmse:0.46753
[1000]	validation_0-rmse:0.46611
[1100]	validation_0-rmse:0.46475
[1200]	validation_0-rmse:0.46394
[1300]	validation_0-rmse:0.46291
[1400]	validation_0-rmse:0.46239
[1500]	validation_0-rmse:0.46163
[1600]	validation_0-rmse:0.46096
[1700]	validation_0-rmse:0.46036
[1800]	validation_0-rmse:0.45992
[1900]	validation_0-rmse:0.45962
[1999]	validation_0-rmse:0.45957

▼ テストデータでのRMSE: 0.4440


ここから、このできたハイパパラメータを利用してDNNとのスタッキングモデルを作っていく

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
import xgboost as xgb 

print("--- Step 1: XGBoostによるOOF予測の作成 ---")
kf = KFold(n_splits=5, shuffle=True, random_state=42)


oof_xgb = np.zeros(len(X_train))
test_xgb = np.zeros(len(X_test))


best_params = study.best_params.copy()
best_params['random_state'] = 42
best_params['eval_metric'] = 'rmse'
best_params['n_estimators'] = 2000

for fold, (train_idx, valid_idx) in enumerate(kf.split(X_train)):
    X_tr, y_tr = X_train[train_idx], y_train[train_idx]
    X_va, y_va = X_train[valid_idx], y_train[valid_idx]

    model = xgb.XGBRegressor(**best_params, early_stopping_rounds=50)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False
    )

    oof_xgb[valid_idx] = model.predict(X_va)
    test_xgb += model.predict(X_test) / kf.n_splits
    
print("OOF予測値の作成完了")

print("\n--- Step 2: DNN用のデータ結合とスケーリング ---")

X_train_combined = np.column_stack((X_train, oof_xgb))#説明変数+OOF予測値
X_test_combined = np.column_stack((X_test, test_xgb))#今はまだ使わない

scaler_dnn_X = StandardScaler()
X_train_dnn_scaled = scaler_dnn_X.fit_transform(X_train_combined)
X_test_dnn_scaled = scaler_dnn_X.transform(X_test_combined)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

print(f"DNN用 学習データ形状: X={X_train_dnn_scaled.shape}, y={y_train_scaled.shape}")

OOF予測値の作成完了
Step 2: DNN用のデータ結合とスケーリング ---
DNN用 学習データ形状: X=(16512, 9), y=(16512,)


In [ ]:
class CascadeDNN(nn.Module):
    def __init__(self, input_dim):
        super(CascadeDNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            #nn.BatchNorm1d(128),    
            nn.ReLU(),
            nn.Dropout(0.3),        
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            
            nn.Linear(32, 1)       
        )

    def forward(self, x):
        return self.net(x)

X_t = torch.FloatTensor(X_train_dnn_scaled)
y_t = torch.FloatTensor(y_train_scaled).unsqueeze(1) 


batch_size = 64
dataset = TensorDataset(X_t, y_t)#束にしている
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

input_dimension = X_train_dnn_scaled.shape[1]
dnn_model = CascadeDNN(input_dim=input_dimension)
criterion = nn.MSELoss() 
optimizer = optim.Adam(dnn_model.parameters(), lr=0.001, weight_decay=1e-4) # L2正則化を追加

print("\n--- Step 3: DNNモデルの準備完了 ---")
print(dnn_model)